Adrian N. Vangen Viktor M.L: Hedman

Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import re


### Task 1,1: File reder function

In [3]:
def filereader(file:str) -> tuple:
    """
    This function take a string contaning the path of the file, and check the encoding used.
    The function checks the encoding used by using a loop tha try the diffrent encodings and 
    stors it in a list to geder wit the contents od the file.
    The function then returns that list contaning the contents of the file and the encoding used.
    The function also assume the only to encodings are utf-8 and latin-1.
    """
    path=Path(file)

    encodings = ('utf-8', 'latin-1')

    for encoding in encodings:
        try:
            with open(path, 'r', encoding = encoding) as file:
                file_text_ontents=file.read()
        except UnicodeDecodeError:
            continue

    return (file_text_ontents, encoding)

print(filereader('Data\\2026\\library_cafe\\POS_01\\register_sales.log'))


('[2026-10-15 07:00:45] [POS_01] item=Espresso price=2.77 USD (SUCCESS)\n[2026-10-15 07:01:11] [POS_01] item=Muffin price=3.86 NOK (CANCELLED)\n=== NETWORK SYNC OK ===\n[2026-10-15 07:04:05] [POS_01] item=Muffin price=3.02 USD (REFUND)\n[2026-10-15 07:05:18] [POS_01] item=Cappuccino price=999.00 USD (SUCCESS)\n[2026-10-15 07:07:38] [POS_01] item=Muffin price=3.00 EUR (REFUND)\n[2026-10-15 07:10:02] [POS_01] item=Sandwich price=7.62 NOK (SUCCESS)\n[2026-10-15 07:10:43] [POS_01] item=Sandwich price=7.79 USD (CANCELLED)\n[2026-10-15 07:13:39] [POS_01] item=Sandwich price=7.36 NOK (SUCCESS)\n[2026-10-15 07:16:05] [POS_01] item=Cappuccino price=4.52 NOK (CANCELLED)\n[2026-10-15 07:16:40] [POS_01] item=Latte price=4.83 USD (SUCCESS)\n=== NETWORK SYNC OK ===\n[2026-10-15 07:17:22] [POS_01] item=Latte price=4.52 NOK (SUCCESS)\n[2026-10-15 07:18:38] [POS_01] item=Latte price=4.87 EUR (CANCELLED)\n[2026-10-15 07:20:37] [POS_01] item=Sandwich price=7.10 USD (SUCCESS)\n[2026-10-15 07:23:13] [POS_0

### Task1,2: Log parser function

In [4]:
file_text=filereader('Data\\2026\\library_cafe\\POS_01\\register_sales.log')[0]

def log_parser(file_text: str) -> list[dict]:
    """
    The function take a the contents of a sale file and returns 
    a list containing dictionarys with valid sales.
    First it split the lines and make an enpty record list.
    Then it define the reguler expressions.
    Lastly the function uses a loop to cheek and 
    store the matching lines as a list of dictionerys.
    """

    file_lines=file_text.splitlines() # Slit the text into lines

    recordslist=[] #Define the list for the record logs.

    #This list of regulare expressions are made using gemini ai
    log_pattern = re.compile(
    r'^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]\s+'
    r'\[(?P<terminal>[A-Za-z0-9_]+)\]\s+'
    r'item=(?P<item>.+?)\s+'
    r'price=(?P<price>\d+(?:\.\d+)?)\s+'
    r'(?P<currency>[A-Z]{3})\s+'
    r'\((?P<status>[A-Z]+)\)$'
    )

    for line in file_lines:

        #Cheek if the line match eny of the regular expressions and stores the match inn grups
        match=log_pattern.match(line)

        #If it match it makes in to a dictionery and appends it in to the recordslist
        if match:
            match_dict=match.groupdict()
            recordslist.append(match_dict)

    return recordslist 

log=log_parser(file_text)
print(log)

[{'timestamp': '2026-10-15 07:00:45', 'terminal': 'POS_01', 'item': 'Espresso', 'price': '2.77', 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:01:11', 'terminal': 'POS_01', 'item': 'Muffin', 'price': '3.86', 'currency': 'NOK', 'status': 'CANCELLED'}, {'timestamp': '2026-10-15 07:04:05', 'terminal': 'POS_01', 'item': 'Muffin', 'price': '3.02', 'currency': 'USD', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:05:18', 'terminal': 'POS_01', 'item': 'Cappuccino', 'price': '999.00', 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:07:38', 'terminal': 'POS_01', 'item': 'Muffin', 'price': '3.00', 'currency': 'EUR', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:10:02', 'terminal': 'POS_01', 'item': 'Sandwich', 'price': '7.62', 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:10:43', 'terminal': 'POS_01', 'item': 'Sandwich', 'price': '7.79', 'currency': 'USD', 'status': 'CANCELLED'}, {'timestamp': '2026-10-15 07:13:39', 'te

### Task 4: Funktions that uses pandas

In [ ]:
def Dataframe_conv(li: list[dict[str, str]]) -> pd.DataFrame:
    """Convert a list of dictionaries into a pandas DataFrame.
    It uses the function DataFrame and uses the function to_date time
    to convert the timestamp values to the date time format. 
    The function also sorts the dataframe by the time in ascending order.
    """

    df=pd.DataFrame(li) # Makes the dataframe using the list
    df['timestamp'] = pd.to_datetime(df['timestamp']) # Convert the timstamps to datetime format
    df['price'] = pd.to_numeric(df['price']) # Convert the price to numbers
    df_shorted = df.sort_values(by='timestamp', ascending = True) # Sorts the dataframe


    return df_shorted

def item_group(df: pd.DataFrame) -> pd.DataFrame:
    """ 
    The Function grops the data by item, resampel the transactions
    into 1 hour windows and shows the total price. 
    First the function converts the price to numbers 
    and set the index to the timstamps.
    Then it grups the data baist on items, sets the data in 1_hour windows
    and calculates the total price. Lastly the grupped data frame is being returned.
    """

    
    df = df.set_index('timestamp')

    df = df.groupby(['item', pd.Grouper('timestamp', freq='1h')])['price'].sum().reset_index()
    return df

def pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """ 
    This function uses the pivot_table function bilt in to pandas to make a pivot tabel.
    The pivot table is maid using the pivot_table function includet in the pandas librery.
    """

    pivot_tab = df.pivot_table(values = 'price', columns = 'item', index = 'status', aggfunc = 'sum')

    return pivot_tab

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    refund_ratio = refund_count/(sales_count + 10**(-6))
    return refund_ratio 

def refund_ratio_df(df: pd.DataFrame) -> pd.DataFrame:
    """ 
    This function uses the refund ratio function vi definf to calculate
    the refundratio to the of the prices per item in the data frame.
    """

    rl=[]
    vl=[]

    for value in list(df['item'].unique()):
        count = df[df['item'] == value]['status'].value_counts()
        ratio = compute_refund_ratio(count['SUCCESS'], count['REFUND'])
        rl.append(ratio)
        vl.append(value)

    d={'item':vl, 'ratio':rl}
    df=pd.merge(df, pd.DataFrame(d), on='item', how='left')
    
    return df

    

df=Dataframe_conv(log)

print(df.head(5))
print('______')
print(item_group(df))
print('______')
print(pivot_table(df))
print(refund_ratio_df(df).head(5))



            timestamp terminal        item   price currency     status
0 2026-10-15 07:00:45   POS_01    Espresso    2.77      USD    SUCCESS
1 2026-10-15 07:01:11   POS_01      Muffin    3.86      NOK  CANCELLED
2 2026-10-15 07:04:05   POS_01      Muffin    3.02      USD     REFUND
3 2026-10-15 07:05:18   POS_01  Cappuccino  999.00      USD    SUCCESS
4 2026-10-15 07:07:38   POS_01      Muffin    3.00      EUR     REFUND
______
          item           timestamp    price
0   Cappuccino 2026-10-15 07:00:00  1007.26
1   Cappuccino 2026-10-15 08:00:00    25.05
2   Cappuccino 2026-10-15 09:00:00  1003.35
3    Croissant 2026-10-15 07:00:00    25.98
4    Croissant 2026-10-15 08:00:00    18.28
5     Espresso 2026-10-15 07:00:00     9.60
6     Espresso 2026-10-15 08:00:00    30.51
7        Latte 2026-10-15 07:00:00    49.62
8        Latte 2026-10-15 08:00:00    17.14
9        Latte 2026-10-15 09:00:00    13.51
10      Muffin 2026-10-15 07:00:00    19.96
11      Muffin 2026-10-15 08:00:00    1

### Task 5